# X-Pand: Model Mechanics & Architecture

This notebook explains the "under the hood" logic of the X-Pand profitability predictor. 

### The Core Problem
We need to predict if a 500m x 500m grid cell in Delhi NCR will be profitable. Standard ML models (like LightGBM) are great at patterns, but they often ignore **geography**—they treat a high-population area in South Delhi the same as one in North Delhi. 

To solve this, X-Pand uses a **Feature Augmentation Pipeline**.

## 1. The Feature Pipeline: 8 → 10

The LightGBM model doesn't just look at raw demographics. It looks at demographics **enhanced by spatial intelligence**.

### Step A: The 8 Base Features
We start with 8 demographic and geospatial features for every cell:
1. `pop_density`
2. `income_index`
3. `competitor_count`
4. `road_density`
5. `transit_stops`
6. `warehouse_proximity_km`
7. `lag_profitable` (spatial lag)
8. `lag_pop_density` (spatial lag)

### Step B: Geographically Weighted Regression (GWR)
Before training the main classifier, we run **GWR**. 

GWR is a "Local" model. Unlike standard regression which finds one set of weights for the whole city, GWR finds **custom weights for every single square** by looking at its neighbors. 

It outputs two critical metrics for every square:
9. **`gwr_intercept`**: The local "baseline" profitability of that specific neighborhood.
10. **`gwr_local_r2`**: A measure of how predictable that specific neighborhood is based on its neighbors.

In [ ]:
import sys
sys.path.insert(0, '..')
import joblib
import pandas as pd
import numpy as np

# Load the models to see the feature counts
lgbm_model = joblib.load('../models/lgbm_model.pkl')
gwr_results = joblib.load('../models/gwr_coeffs.pkl')

# Check LightGBM expected features
base_estimator = lgbm_model.calibrated_classifiers_[0].estimator
print(f"LightGBM expects: {base_estimator.n_features_in_} features")
print(f"Feature names in model: {base_estimator.feature_name_}")

print(f"\nGWR Params shape: {gwr_results.params.shape} (12987 cells, 9 coefficients)")

## 2. Visualizing the Data Flow

```mermaid
graph TD
    A[8 Base Features] --> B[GWR Model]
    B --> C[gwr_intercept]
    B --> D[gwr_local_r2]
    A --> E[Final Feature Matrix]
    C --> E
    D --> E
    E --> F[LightGBM Classifier]
    F --> G[Profitability Probability]
```

## 3. Why this "GWR → LightGBM" flow matters

This is called **Spatial Feature Engineering**. By feeding the GWR outputs into LightGBM, we are essentially "pre-digesting" the geography for the model.

*   **Without GWR:** LightGBM might see `pop_density=9000` and think it's always good. 
*   **With GWR:** LightGBM sees `pop_density=9000` AND `gwr_intercept=-0.45`. It now understands that *even though population is high, the local neighborhood baseline is low*, so it should be more cautious.

### Importance of Synchronization
> **CRITICAL:** Because the LightGBM model was trained on 10 features, it will **crash** if you only give it the 8 base features. This is why the API (`api/main.py`) must extract the GWR features for any new location before asking LightGBM for a prediction.

## 4. SHAP: Understanding the "Why"

Finally, we use **SHAP** to explain which of these 10 features pushed the prediction up or down. You can see this in the **Cell Detail** tab of the dashboard.